# M65 A2 preservation control

Revision **M65-A2-PRESERVATION-2026-10-05-r1**. Run sections **1–6 in order** on an A100 CUDA 13.0 runtime. This is a new, self-contained notebook; do not run historical M62/M63 recovery cells. No iPhone is needed.

M64 successfully reproduced unchanged A2 and the MonoPRIO candidate, but the released prior bank lacks construction IDs and differs from the default builder recipe. This notebook records that audit and trains **one epoch of GT plus original-A2 preservation**, with no MonoPRIO supervision. Keep both product classes. The original A2 checkpoint is never overwritten.

Deliverable: `m65_results.zip` with the prior audit, fresh baseline, zero-update smoke, one-epoch training summary, full-set before/after metrics, and the fixed preservation gate. Stop for review afterward; this notebook cannot authorize external-teacher KD or deployment.

## 1 Settings and project update

The default M64 path identifies the reviewed r2 run. Keep it unchanged. A changed GPU or runtime requires a fresh `RUN_ID`; preserve earlier output folders. All commands use a private interpreter and durable logs.

In [ ]:
# M65-A2-PRESERVATION-2026-10-05-r1 — self-contained one-epoch control
from pathlib import Path
from collections import deque
import json, os, shlex, subprocess, sys, runpy
from google.colab import drive
drive.mount('/content/drive')

REVISION = 'M65-A2-PRESERVATION-2026-10-05-r1'
RUN_ID = 'm65_a2_preservation_r1'
MOBILE_REPO = Path('/content/mobile_adas3d')
A2_REPO = Path('/content/MonoDETR_M65_A2')
TEACHER_SOURCE = Path('/content/MonoPRIO_M65_AUDIT')
M64_OUTPUT = Path('/content/drive/MyDrive/mobile_adas3d_outputs/students/monodetr_m64_teacher_qualification/m64_teacher_qualification_r2')
OUTPUT = Path('/content/drive/MyDrive/mobile_adas3d_outputs/students/monodetr_m65_a2_preservation') / RUN_ID
DATASET = Path('/content/kitti_m65')
SPLITS = Path('/content/drive/MyDrive/mobile_adas3d_splits/kitti_chen')
PYTHON = Path('/content/m65_cuda130_venv/bin/python')
CUDA = Path('/usr/local/cuda-13.0')
if not (CUDA/'bin/nvcc').is_file(): CUDA = Path('/usr/local/cuda')
MANIFEST = OUTPUT/'m65_manifest.json'
SETUP_READY = MANIFEST_READY = TRAIN_READY = TRAIN_COMPLETE = False
ENV = os.environ.copy()
OUTPUT.mkdir(parents=True, exist_ok=True)
print(REVISION, '\nOutput:', OUTPUT, flush=True)

def run_logged(command, name, cwd=None):
    command = [str(x) for x in command]
    log_path = OUTPUT/'colab_logs'/f'{name}.log'
    log_path.parent.mkdir(parents=True, exist_ok=True)
    tail = deque(maxlen=60)
    print('+', shlex.join(command), '\nDurable log:', log_path, flush=True)
    with log_path.open('a') as log:
        log.write('\nCOMMAND: '+shlex.join(command)+'\n'); log.flush()
        with subprocess.Popen(command, cwd=str(cwd) if cwd else None, env=ENV, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1) as p:
            for line in p.stdout:
                print(line, end='', flush=True); log.write(line); log.flush(); tail.append(line.rstrip())
            rc = p.wait()
        log.write(f'\nEXIT CODE: {rc}\n')
    if rc: raise RuntimeError(f'Exit {rc}; full log={log_path}\n'+'\n'.join(tail))

def checkout(path, url, commit):
    if '[' in url or ']' in url or not url.startswith('https://github.com/'):
        raise ValueError('Use a plain public GitHub URL, not Markdown')
    if not path.exists():
        run_logged(['git','clone','--no-checkout',url,path],f'clone_{path.name}')
        run_logged(['git','checkout','--detach',commit],f'pin_{path.name}',path)
    if not (path/'.git').is_dir(): raise RuntimeError(f'Preserve {path}; choose a fresh checkout path')
    head = subprocess.check_output(['git','rev-parse','HEAD'],cwd=path,text=True).strip()
    origin = subprocess.check_output(['git','remote','get-url','origin'],cwd=path,text=True).strip()
    if head != commit or origin.rstrip('/') != url:
        raise RuntimeError(f'Unexpected checkout identity: {path}; do not reset it')

def workflow(stage, *extra):
    command = [PYTHON,'-X','faulthandler','-u',MOBILE_REPO/'scripts/m65_a2_preservation_control.py',stage]
    command += ['--output',OUTPUT] if stage in ('prepare','bundle') else ['--manifest',MANIFEST]
    run_logged(command+list(extra),'m65_'+stage+'_'+'_'.join(str(x).replace('/','_') for x in extra if not str(x).startswith('/')),MOBILE_REPO)

if not MOBILE_REPO.exists():
    run_logged(['git','clone','https://github.com/Ali-RT/mobile_adas3d.git',MOBILE_REPO],'clone_mobile')
branch = subprocess.check_output(['git','branch','--show-current'],cwd=MOBILE_REPO,text=True).strip()
if branch != 'main': raise RuntimeError(f'Use main, not {branch!r}; preserve other branch work')
run_logged(['git','pull','--ff-only'],'update_mobile',MOBILE_REPO)
SCRIPT = MOBILE_REPO/'scripts/m65_a2_preservation_control.py'
if not SCRIPT.is_file(): raise FileNotFoundError('M65 is not present. Reload the published main-branch notebook.')
sys.path.insert(0,str(MOBILE_REPO/'scripts'))
namespace = runpy.run_path(str(SCRIPT),run_name='m65_revision_check')
if namespace['REVISION'] != REVISION: raise RuntimeError('Notebook/script revisions differ; reload the latest notebook')


## 2 Working isolated runtime and fresh sources

Reuse the successful M64 CUDA 13.0 setup/build helpers in a new M65 environment and checkout. Compile only A2. MonoPRIO source is cloned for a CPU-only provenance audit; its model is not run or trained.

In [ ]:
SETUP_READY = MANIFEST_READY = TRAIN_READY = TRAIN_COMPLETE = False
if not SCRIPT.is_file(): raise RuntimeError('Run section 1 first')
helpers = runpy.run_path(str(MOBILE_REPO/'scripts/setup_m64_runtime.py'),run_name='shared_cuda13_helpers')
ENV = helpers['runtime_env'](CUDA,PYTHON.parent)
run_logged([sys.executable,'-u',MOBILE_REPO/'scripts/setup_m64_runtime.py','--venv',PYTHON.parent.parent,'--cuda',CUDA,'--receipt',OUTPUT/'m65_runtime_receipt.json'],'m65_runtime',MOBILE_REPO)
checkout(A2_REPO,'https://github.com/ZrrSkywalker/MonoDETR.git','6994b9f512400b258c6edb75f77423beb9c126f2')
checkout(TEACHER_SOURCE,'https://github.com/Leon-Davies/MonoPRIO.git','884b7d8562e528031616c4773170bfc4fe211bf0')
for patch in ('patch_monodetr_colab_compat.py','patch_monodetr_product_taxonomy.py','patch_monodetr_mobilenetv4.py'):
    run_logged([PYTHON,MOBILE_REPO/'scripts'/patch,'--monodetr-repo',A2_REPO],patch.removesuffix('.py'),MOBILE_REPO)
run_logged([PYTHON,MOBILE_REPO/'scripts/patch_m64_inference_sources.py','--repo',A2_REPO,'--role','a2'],'m65_patch_a2',MOBILE_REPO)
run_logged([PYTHON,'-X','faulthandler','-u',MOBILE_REPO/'scripts/build_m64_attention.py','--repo',A2_REPO,'--role','a2'],'m65_build_a2',MOBILE_REPO)
for test in ('test_m64_teacher_qualification.py','test_m65_preservation_control.py'):
    run_logged([PYTHON,'-m','unittest','discover','-s','tests','-p',test,'-v'],test.removesuffix('.py'),MOBILE_REPO)
SETUP_READY = True
print('M65 section 2 passed: new A2-only build, shared successful CUDA13 recipe.')


## 3 Data and reviewed M64 evidence

Restore original KITTI images, labels, calibration and Chen 3712/3769 splits. The exact reviewed M64 report and A2 checkpoint must remain on Drive. M65 records a new identity rather than rewriting M64, M62 or M63.

In [ ]:
MANIFEST_READY = TRAIN_READY = TRAIN_COMPLETE = False
if not SETUP_READY: raise RuntimeError('Run sections 1 and 2 first')
for name in ('m64_manifest.json','m64_teacher_qualification.json'):
    if not (M64_OUTPUT/name).is_file(): raise FileNotFoundError(M64_OUTPUT/name)
candidates = ['/content/kitti_m64','/content/kitti_m62','/content/kitti_m61','/content/kitti','/content/monodetr_kitti','/content/monodetr_kitti_a2','/content/drive/MyDrive/datasets/kitti']
command = [PYTHON,MOBILE_REPO/'scripts/restore_m63_data.py','--dataset',DATASET,'--splits',SPLITS]
for candidate in candidates: command += ['--candidate',candidate]
run_logged(command,'m65_restore_data',MOBILE_REPO)
workflow('prepare','--m64-output',M64_OUTPUT,'--a2-repo',A2_REPO,'--dataset-root',DATASET)
m = json.loads(MANIFEST.read_text())
assert m['recipe']['epochs'] == 1 and m['kd_authorized'] is False
MANIFEST_READY = True
print('M65 section 3 passed. Original A2 weights and exact split identities verified.')


## 4 Prior audit and fresh A2 baseline and smoke

The prior report is expected to leave external KD disabled. It does not block this teacher-independent control. Evaluate unchanged A2 on all 3769 images before any updates, then test GT backward, reliable object matching and preservation gradients with zero optimizer steps. Do not bypass a failed baseline or smoke.

In [ ]:
if not MANIFEST_READY: raise RuntimeError('Run sections 1–3 first')
TRAIN_READY = False
old = json.loads((M64_OUTPUT/'m64_manifest.json').read_text())
PRIOR = Path(old['asset_receipt']['prior']['path'])
if not PRIOR.is_file(): raise FileNotFoundError(PRIOR)
run_logged([PYTHON,MOBILE_REPO/'scripts/audit_m65_prior_provenance.py','--teacher-repo',TEACHER_SOURCE,'--prior',PRIOR,'--output',OUTPUT/'m65_prior_provenance.json'],'m65_prior_audit',MOBILE_REPO)
workflow('infer','--model','baseline')
workflow('metrics','--model','baseline')
workflow('baseline')
workflow('smoke')
smoke = json.loads((OUTPUT/'m65_training_smoke.json').read_text())
assert smoke['passed'] and smoke['optimizer_steps'] == 0 and not smoke['external_kd_enabled']
TRAIN_READY = True
print('Baseline and CUDA smoke passed. Only the fixed one-epoch A2-preservation control may run.')


## 5 One epoch of GT and original A2 preservation

928 updates, batch4, fresh AdamW, LR1e-6, no AMP or mixup. Freeze BatchNorm running statistics and affine weights; disable dropout. Native GT losses supervise both classes. A frozen original-A2 copy sees each identical transformed image online. Match both inference groups independently to GT and preserve only reliable components, balanced within class. Log losses and pair counts every20 batches.

Interruption before a completed epoch restarts this one epoch from original A2; no partial optimizer state is silently reused. A completed atomic checkpoint is reused with **zero additional updates**. This is not a full run or stronger-teacher KD.

In [ ]:
if not TRAIN_READY: raise RuntimeError('Run sections 1–4; do not bypass failed preflight')
TRAIN_COMPLETE = False
workflow('train')
summary = json.loads((OUTPUT/'m65_training_summary.json').read_text())
assert summary['complete'] and summary['completed_epochs'] == 1 and summary['optimizer_steps'] == 928
assert summary['running_buffers_unchanged'] and summary['anchor_unchanged'] and not summary['external_kd_enabled']
TRAIN_COMPLETE = True
print('One-epoch control finished. Now run section 6; do not add epochs or enable KD.')


## 6 Full validation and result bundle

Evaluate the complete split with unchanged product thresholds, top50 and geometry diagnostics. The control-stability gate allows at most0.15 AP points loss in each of four metrics and0.005 absolute nearby-recall loss per class. Report historical accuracy gates separately; they are not lowered. Neither a stable control nor a passing product metric automatically selects this checkpoint or authorizes KD.

Return `m65_results.zip`. This final cell can also bundle partial logs after a failure, after section1 has defined the output directory.

In [ ]:
if TRAIN_COMPLETE:
    workflow('infer','--model','control')
    workflow('metrics','--model','control')
    workflow('review')
else:
    print('Bundling partial diagnostics only. No completed control or training authorization is claimed.')
workflow('bundle')
print('Return:',OUTPUT/'m65_results.zip')
print('STOP FOR REVIEW. No teacher KD, extra epochs, compression, or phone deployment.')
